# Understanding the Governed Data World Pt1

Before proceeding with any analysis, we need to first understand what datasets we are dealing with. And refering to `analysis\dev_full_offline_investigation\surface_atlas.md`, we can see 7 classes:

Surface classes
- `traffic`: canonical behavioural traffic eligible for platform ingestion
- `context`: join or enrichment surface used to understand traffic
- `truth`: offline outcome or supervisory surface
- `telemetry`: audit or operational evidence, not business traffic


Time-safety classes
- `rtdl_safe`: eligible as decision-time context if used correctly
- `offline_only`: batch, truth, or closure-dependent surface; never live-decision input
- `not_traffic`: should not be treated as business traffic even if event-like

But first, we will examine the data as it moves from Layer-1 to Layer-3 so as to get an understanding of it before we begin investigating the surface classes.

Here is the [Layer-Layer view of Data Engine](../../../../docs/design/data-engine/data-engine-segment-synthesis-flow.png)

![data-engine-layer-layer-view](../../../../docs/design/data-engine/data-engine-segment-synthesis-flow.png)


At a high level, the Data Engine moves from **world construction** to **activity realisation** to **behaviour, fraud, and truth**.

- **Layer 1 (`1A` to `3B`) builds the governed world.**
  - `1A` decides the merchant and outlet footprint: how many outlets exist and in which countries.
  - `1B` places those outlets into concrete geography.
  - `2A` assigns civil time zones and DST/legal-time behaviour.
  - `2B` builds the routing fabric that later traffic can use.
  - `3A` refines merchants across intra-country zones where needed.
  - `3B` adds the virtual/edge world for merchants that operate through network edges rather than physical sites.

- **Layer 2 (`5A`, `5B`) brings that world to life over time.**
  - `5A` creates intensity shapes: when and how strongly activity should happen across merchants, classes, channels, zones, and scenarios.
  - `5B` turns those intensities into actual arrival skeletons, producing the first event-like activity surface, `arrival_events_5B`.

- **Layer 3 (`6A`, `6B`) turns arrivals into an entity-rich behavioural world with fraud and truth.**
  - `6A` builds the entity layer: parties, accounts, instruments, devices, IPs, and fraud-role posture.
  - `6B` attaches those entities to arrivals, generates behavioural event streams and flow anchors, overlays fraud campaigns, and then emits the truth products: event labels, flow truth, bank view, and case timelines.

The important interpretation is this: the engine first creates the **world**, then the **clock of activity**, then the **actors and behaviour**, and finally the **fraud/case truth** that the platform consumes offline or operationally. Also, `4A/4B` are not standalone production layers now; their concerns were folded in as cross-cutting gate, validation, and audit disciplines across the earlier segments. See [data_engine_interface.md](/c:/Users/LEGION/Documents/Data%20Science/Python%20&%20R%20Scripts/fraud-detection-system/docs/model_spec/data-engine/interface_pack/data_engine_interface.md), [narrative_1A-to-3B.md](/c:/Users/LEGION/Documents/Data%20Science/Python%20&%20R%20Scripts/fraud-detection-system/docs/model_spec/data-engine/layer-1/narrative/narrative_1A-to-3B.md), and [narrative_5A-and-5B.md](/c:/Users/LEGION/Documents/Data%20Science/Python%20&%20R%20Scripts/fraud-detection-system/docs/model_spec/data-engine/layer-2/narrative/narrative_5A-and-5B.md).


## Notebook Setup

In [1]:
from pathlib import Path
import os

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
import seaborn as sns

# Widen the notebook display so summary tables are easier to inspect while reading.
pd.set_option("display.max_columns", 200)
pd.set_option("display.max_rows", 200)
pd.set_option("display.width", 220)


In [2]:
# Walk upward from the current directory until we hit the repo root.
# We identify the root by the presence of AGENTS.md and the reference folder.
def find_repo_root(start: str | None = None) -> Path:
    here = Path(start or os.getcwd()).resolve()
    for candidate in [here, *here.parents]:
        if (candidate / "AGENTS.md").exists() and (candidate / "reference").exists():
            return candidate
    raise FileNotFoundError("Could not locate repo root")


# Resolve the repo root once so every later path can be expressed relative to it.
ROOT = find_repo_root()
# Point to the exact merchant-universe parquet sealed into the current investigation basis.
transaction_schema_merchant_ids_path = ROOT / "reference" / "layer1" / "transaction_schema_merchant_ids" / "2026-01-03" / "transaction_schema_merchant_ids.parquet"
# Keep the associated bootstrap policy nearby because it is part of the sealed input story.
transaction_schema_bootstrap_policy_path = ROOT / "config" / "layer1" / "1A" / "ingress" / "transaction_schema_merchant_ids.bootstrap.yaml"

merchant_ids_df = pd.read_parquet(transaction_schema_merchant_ids_path)

# Echo the exact artefacts we are analysing so the notebook remains tied to the sealed run inputs.
print(f"transaction_schema_merchant_ids_path: {transaction_schema_merchant_ids_path}")
print(f"bootstrap_policy_path: {transaction_schema_bootstrap_policy_path}")
print(f"shape: {merchant_ids_df.shape}")


transaction_schema_merchant_ids_path: C:\Users\LEGION\Documents\Data Science\Python & R Scripts\fraud-detection-system\reference\layer1\transaction_schema_merchant_ids\2026-01-03\transaction_schema_merchant_ids.parquet
bootstrap_policy_path: C:\Users\LEGION\Documents\Data Science\Python & R Scripts\fraud-detection-system\config\layer1\1A\ingress\transaction_schema_merchant_ids.bootstrap.yaml
shape: (10000, 4)


## Layer 1
[view-img](../../../../docs/design/data-engine/layer-1/layer-1-state-synthesis-flow.png)
![layer-1-view](../../../../docs/design/data-engine/layer-1/layer-1-state-synthesis-flow.png)

### Segment 1A

> "`1A` is the segment that turns a sealed merchant world into a certified outlet-world authority through deterministic lineage control, branch-gated stochastic realism, explicit cross-border order authority, deterministic country-weight and allocation bridges, fingerprint-scoped egress, and final replay validation."

1A Synthesis doc: [here](../../../../docs/design/data-engine/layer-1/1A/1A-state-synthesis.md)

![1A-overview-design-flow.png](../../../../docs/design/data-engine/layer-1/1A/1A-overview-design-flow.png)
[1A-overview-design-flow.png](../../../../docs/design/data-engine/layer-1/1A/1A-overview-design-flow.png)

#### Segment 1A.S0

![1A-S0-design-flow](../../../../docs/design/data-engine/layer-1/1A/1A-S0-design-flow.png)
[1A-S0-design-flow](../../../../docs/design/data-engine/layer-1/1A/1A-S0-design-flow.png)

1A.S0:
> "So `S0` is best understood as the lineage and precompute authority of `1A`, not just a preamble."

For `1A`, `S0` is not the whole segment. It does three things:
- seals the canonical external data universe
- opens and hashes the governed artefacts that define the lawful `1A` run
- emits the sealed `1A` input world and a small set of prep or diagnostic surfaces

A: **External data brought in at `S0.1`**
- `transaction_schema_merchant_ids`: ingress merchant universe
- `iso3166_canonical_2024`: canonical ISO country list
- `world_bank_gdp_per_capita_20250415`: pinned GDP-per-capita vintage
- `gdp_bucket_map_2024`: pinned GDP bucket map

B: **Governed artefacts opened at `S0.2`**
- `hurdle_coefficients.yaml`
- `nb_dispersion_coefficients.yaml`
- `crossborder_hyperparams.yaml`
- `ccy_smoothing_params.yaml`
- `s6_selection_policy.yaml`
- `policy.s3.rule_ladder.yaml`
- `policy.s3.base_weight.yaml` when enabled
- `policy.s3.thresholds.yaml` when referenced
- `transaction_schema_merchant_ids_bootstrap_policy`
- `settlement_shares_2024Q4`
- `ccy_country_shares_2024Q4`
- `numeric_policy_profile`
- `math_profile_manifest`
- optional `static.currency_to_country.map.json` when referenced by policy

C: **Datasets produced by `S0`**
- `sealed_inputs_1A`
- `s0_gate_receipt_1A`
- `crossborder_eligibility_flags`
- optional `crossborder_features`
- optional `hurdle_pi_probs`
- optionally materialised `hurdle_design_matrix`

What matters for the investigation at this stage is the distinction:
- `S0.1` tells us what external data world `1A` begins from
- `S0.2` tells us which governed model, policy, and numeric artefacts define the lawful run
- the `S0` outputs tell us what sealed world later `1A` states are allowed to consume


A.1: `transaction_schema_merchant_ids`
- version: `2026-01-03`
- path: `reference/layer1/transaction_schema_merchant_ids/2026-01-03/transaction_schema_merchant_ids.parquet`
- sha256: `df2f940f061a09a018e133828c9b7b2e7e442a1052b5c0105ef927e2b7c696e8`
- associated bootstrap policy: `config/layer1/1A/ingress/transaction_schema_merchant_ids.bootstrap.yaml`
- bootstrap policy sha256: `f99aace1899982bda221596b64fecd6ff240dd62d8be081159eb1c457765c6c7`

##### Initial analysis of `transaction_schema_merchant_ids`

We start with a bounded statistical summary of the merchant universe:
- shape
- schema
- nulls
- uniqueness
- distributions across `channel`, `home_country_iso`, and `mcc`


In [3]:
# Show a few rows first so we can see the raw row shape before computing summaries.
merchant_ids_df.head(10)

,merchant_id,mcc,channel,home_country_iso
0,1694081450446801,5561,card_present,LU
1,2697116941085629,4789,card_not_present,LU
2,3361754771680650,7996,card_present,ES
3,9012167658511518,4411,card_present,PF
4,10766668736889901,7631,card_present,JP
5,12629669232507242,8062,card_not_present,MC
6,14291053214586959,7699,card_not_present,FO
7,18638034984233531,4225,card_present,EC
8,19498508446557154,7534,card_present,IL
9,23202490577169710,7299,card_present,CA


In [4]:
merchant_ids_df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 10000 entries, 0 to 9999
Data columns (total 4 columns):
 #   Column            Non-Null Count  Dtype 
---  ------            --------------  ----- 
 0   merchant_id       10000 non-null  uint64
 1   mcc               10000 non-null  int32 
 2   channel           10000 non-null  object
 3   home_country_iso  10000 non-null  object
dtypes: int32(1), object(2), uint64(1)
memory usage: 273.6+ KB


Build a compact column-level profile.

In [5]:
# This lets us see data types, null exposure, and cardinality in one table.
pd.DataFrame(
    {
        # Column names in their dataset order.
        "column": merchant_ids_df.columns,
        # Pandas dtypes tell us how the parquet landed in memory.
        "dtype": merchant_ids_df.dtypes.astype(str).values,
        # Absolute null counts highlight missingness immediately.
        "null_count": merchant_ids_df.isna().sum().values,
        # Null percentage makes the counts comparable across columns.
        "null_pct": (merchant_ids_df.isna().mean().mul(100).round(2)).values,
        # Distinct counts show the effective breadth of each field.
        "distinct_count": merchant_ids_df.nunique(dropna=False).values,
    }
)

,column,dtype,null_count,null_pct,distinct_count
0,merchant_id,uint64,0,0.0,10000
1,mcc,int32,0,0.0,290
2,channel,object,0,0.0,2
3,home_country_iso,object,0,0.0,190


Capture the first structural checks for the dataset.

In [6]:
# These tell us how big the merchant universe is and whether merchant_id behaves like a true key.
pd.DataFrame(
    [
        # Total row count in the merchant universe.
        {"check": "row_count", "value": len(merchant_ids_df)},
        # Distinct merchant IDs should match row count if merchant_id is unique.
        {"check": "merchant_id_distinct", "value": merchant_ids_df["merchant_id"].nunique()},
        # Duplicate merchant IDs would be an immediate structural concern for this ingress table.
        {"check": "merchant_id_duplicate_rows", "value": int(merchant_ids_df["merchant_id"].duplicated().sum())},
        # Number of represented home countries.
        {"check": "country_distinct", "value": merchant_ids_df["home_country_iso"].nunique()},
        # Number of represented MCC categories.
        {"check": "mcc_distinct", "value": merchant_ids_df["mcc"].nunique()},
        # Number of channel values present in the dataset.
        {"check": "channel_distinct", "value": merchant_ids_df["channel"].nunique()},
    ]
)

,check,value
0,row_count,10000
1,merchant_id_distinct,10000
2,merchant_id_duplicate_rows,0
3,country_distinct,190
4,mcc_distinct,290
5,channel_distinct,2


In [7]:
# Count merchants by channel to see the overall CP vs CNP split.
merchant_ids_df["channel"].value_counts(dropna=False).rename_axis("channel").reset_index(name="merchant_count")

,channel,merchant_count
0,card_present,7473
1,card_not_present,2527


List the most represented home countries.

In [8]:
# We limit to the top 20 so the first read stays compact.
merchant_ids_df["home_country_iso"].value_counts(dropna=False).head(20).rename_axis("home_country_iso").reset_index(name="merchant_count")

,home_country_iso,merchant_count
0,MC,800
1,BM,436
2,LU,379
3,IE,337
4,GH,315
5,CH,304
6,NO,262
7,SG,238
8,AU,212
9,US,211


Measure how much of the full merchant universe is captured by the leading countries.

In [11]:
# Rank countries by merchant count so we can see how quickly the merchant universe concentrates.
country_concentration_df = merchant_ids_df["home_country_iso"].value_counts(dropna=False).rename_axis("home_country_iso").reset_index(name="merchant_count")

# Convert raw counts into percentage share of the full merchant universe.
country_concentration_df["merchant_share_pct"] = (country_concentration_df["merchant_count"] / len(merchant_ids_df) * 100).round(2)

# Cumulative share shows how much of the universe is covered as we move down the ranked list.
country_concentration_df["cumulative_share_pct"] = country_concentration_df["merchant_share_pct"].cumsum().round(2)

# Inspect the leading rows first, then read off top-5, top-10, top-20 style coverage from the cumulative column.
country_concentration_df.head(20)

,home_country_iso,merchant_count,merchant_share_pct,cumulative_share_pct
0,MC,800,8.00,8.00
1,BM,436,4.36,12.36
2,LU,379,3.79,16.15
3,IE,337,3.37,19.52
4,GH,315,3.15,22.67
5,CH,304,3.04,25.71
6,NO,262,2.62,28.33
7,SG,238,2.38,30.71
8,AU,212,2.12,32.83
9,US,211,2.11,34.94


> Interesting most of our merchants are in Monaco? Then Bermuda? Lol \
> But why though?

List the most represented MCC codes.

In [9]:
# This shows whether the merchant universe is broadly spread or heavily concentrated in a few categories.
merchant_ids_df["mcc"].value_counts(dropna=False).head(20).rename_axis("mcc").reset_index(name="merchant_count")

,mcc,merchant_count
0,8651,52
1,7922,52
2,5992,51
3,5193,50
4,5937,49
5,7299,49
6,7011,47
7,4812,47
8,5521,47
9,7801,47


Measure how much of the full merchant universe is captured by the leading MCC codes.

In [12]:
# Rank MCCs by merchant count so we can compare category concentration with country concentration.
mcc_concentration_df = merchant_ids_df["mcc"].value_counts(dropna=False).rename_axis("mcc").reset_index(name="merchant_count")

# Express each MCC's count as a share of the full merchant universe.
mcc_concentration_df["merchant_share_pct"] = (mcc_concentration_df["merchant_count"] / len(merchant_ids_df) * 100).round(2)

# Cumulative share shows how quickly the top MCCs accumulate coverage.
mcc_concentration_df["cumulative_share_pct"] = mcc_concentration_df["merchant_share_pct"].cumsum().round(2)

# Inspect the leading rows first, then read off top-5, top-10, top-20 style coverage from the cumulative column.
mcc_concentration_df.head(20)

,mcc,merchant_count,merchant_share_pct,cumulative_share_pct
0,8651,52,0.52,0.52
1,7922,52,0.52,1.04
2,5992,51,0.51,1.55
3,5193,50,0.50,2.05
4,5937,49,0.49,2.54
5,7299,49,0.49,3.03
6,7011,47,0.47,3.50
7,4812,47,0.47,3.97
8,5521,47,0.47,4.44
9,7801,47,0.47,4.91


In [10]:
# First identify the countries with the largest merchant counts.
top_countries = merchant_ids_df["home_country_iso"].value_counts().head(15).index
# Then cross-tab those countries against channel to inspect country-level channel mix.
pd.crosstab(
    merchant_ids_df.loc[merchant_ids_df["home_country_iso"].isin(top_countries), "home_country_iso"],
    merchant_ids_df.loc[merchant_ids_df["home_country_iso"].isin(top_countries), "channel"],
).sort_index()

channel,card_not_present,card_present
home_country_iso,,
AU,53,159
BM,109,327
CH,76,228
DK,50,150
FO,46,139
GH,79,236
IE,84,253
IS,46,139
LU,95,284
